In [ ]:
# IMPORTS
import json

import pandas as pd
from emag_reports.logger import logger
from emag_reports.models.prediction import PredictionSettings
from emag_reports.models.prediction import SurvivalScoring
from emag_reports.prediction.summary import SurvivalSummarizer
from emag_reports.prediction.survival import SurvivalManager
from emag_reports.preprocess import impute_missing_values
from emag_reports.preprocess import preprocess_survival_data
from emag_reports.preprocess import remove_low_quality_attributes
from emag_reports.preprocess import SurvivalStratifiedKFold
from emag_reports.preprocess import SurvivalStratifiedShuffleSplit
from IPython.display import display
from IPython.display import HTML
from IPython.display import Markdown as md
from itables import show

In [ ]:
## DEFAULT PARAMETERS
# DATASET PARAMETERS
TITLE = "Predictive analysis report - survival"
NAME = "bone marrow"
CLASS_ATTRIBUTE = json.dumps({"event": "survival_status", "time": "survival_time"})
# DATA PREPARATION
# Unimportant attributes elimination
UNIMPORTANT_ATTRIBUTE_ELIMINATION = True
I = 0.9
S = 0.9
M = 0.9
T = 0.9
# Missing values imputation
MISSING_VALUES_IMPUTATION = True
# Strategy for missing values imputation
STRATEGY_NUMERICAL = 'mean'
STRATEGY_NOMINAL = 'most_frequent'
# ALGORITHMS PARAMETERS
ANALYSIS_MODE = PredictionSettings.AnalysisMode.TRAIN_TEST
TEST_SIZE = 0.3
# GRID SEARCH
GRID_SEARCH = True
SCORING = SurvivalScoring.C_INDEX
# INCLUDED MODELS
COX_PROPORTIONAL_HAZARD = True
DECISION_TREE = True
RANDOM_FOREST = True
SVM = True
GRADIENT_BOOSTING = True

DATASET = None

In [ ]:
# UNPACK VALUES
CLASS_ATTRIBUTE = json.loads(CLASS_ATTRIBUTE)
CLASS_ATTRIBUTE_EVENT, CLASS_ATTRIBUTE_TIME = CLASS_ATTRIBUTE["event"], CLASS_ATTRIBUTE["time"]
# PATCH PARAMETERS WHICH HAVE REQUIRED VALUES FOR SURVIVAL
USE_LOW_CORRELATION_ATTRIBUTE_REMOVAL = False # always False for survival analysis

In [ ]:
display_analysis_mode = "train-test split" if ANALYSIS_MODE == PredictionSettings.AnalysisMode.TRAIN_TEST else "5-fold cross-validation"
html_content = f"""
<div style='text-align: center; margin-bottom: 30px; font-family: Arial, sans-serif;'>
    <h1 style='font-size: 2em; color: #2c3e50;'><strong>Predictive analysis report</strong></h1>
    <h2 style='font-size: 1.3em; color: #34495e;'>Evaluate the predictive ability of various models</h2>
</div>
<div style='margin: 0 auto; width: 90%; font-family: Arial, sans-serif; color: #2c3e50;'>
    <section style='margin-bottom: 30px;'>
        <h2 style='border-bottom: 2px solid #7f8c8d; padding-bottom: 10px;'>Report information</h2>
        <div style='padding-left: 2em; line-height: 1.5;'>
            <p><strong>Report name:</strong> <span style='font-style: italic; color: #2980b9;'>{TITLE}</span></p>
            <p><strong>Dataset name:</strong> <span style='font-style: italic; color: #2980b9;'>{NAME}</span></p>
        </div>
    </section>
    <section style='margin-bottom: 30px;'>
        <h3 style='border-bottom: 1px solid #bdc3c7; padding-bottom: 5px;'>Data preprocessing parameters</h3>
        <div style='padding-left: 2em; line-height: 1.5;'>
            <p>Model evaluation method: <span style='font-style: italic; color: #2980b9;'>{display_analysis_mode}</span></p>"""

if ANALYSIS_MODE == PredictionSettings.AnalysisMode.TRAIN_TEST:
    html_content += f"""
            <p>Test set size: <span style='font-style: italic; color: #2980b9;'>{TEST_SIZE}</span></p>"""

html_content += f"""
            <p>Perform a search of hyperparameter space: <span style='font-style: italic; color: #2980b9;'>{GRID_SEARCH}</span></p>
            <p>Remove attributes of low quality: <span style='font-style: italic; color: #2980b9;'>{UNIMPORTANT_ATTRIBUTE_ELIMINATION}</span></p>"""

if UNIMPORTANT_ATTRIBUTE_ELIMINATION:
    html_content += f"""
            <div style='padding-left: 2em;'>
                <p>Excessive number of unique values threshold: <span style='font-style: italic; color: #2980b9;'>{I}</span></p>
                <p>Stability threshold: <span style='font-style: italic; color: #2980b9;'>{S}</span></p>
                <p>Missing values threshold: <span style='font-style: italic; color: #2980b9;'>{M}</span></p>
                <p>Text-like - the variable values resemble text threshold: <span style='font-style: italic; color: #2980b9;'>{T}</span></p>
            </div>"""

html_content += f"""
            <p>Replace missing values: <span style='font-style: italic; color: #2980b9;'>{MISSING_VALUES_IMPUTATION}</span> (always true for survival problems)</p>"""
if MISSING_VALUES_IMPUTATION:
    html_content += f"""
            <div style='padding-left: 2em;'>
                <p>For numeric attributes by: <span style='font-style: italic; color: #2980b9;'>{STRATEGY_NUMERICAL}</span></p>
                <p>For nominal attributes by: <span style='font-style: italic; color: #2980b9;'>{STRATEGY_NOMINAL}</span></p>
            </div>
        </div>
    </section>"""

html_content += f"""
    <section style='margin-bottom: 30px;'>
        <h3 style='border-bottom: 1px solid #bdc3c7; padding-bottom: 5px;'>Selected models</h3>
        <div style='padding-left: 2em; line-height: 1.5;'>
            <p>Cox proportional hazard (linear model): <span style='font-style: italic; color: #2980b9;'>{COX_PROPORTIONAL_HAZARD}</span></p>
            <p>Trees: <span style='font-style: italic; color: #2980b9;'>{DECISION_TREE}</span></p>
            <p>Support vector machine: <span style='font-style: italic; color: #2980b9;'>{SVM}</span></p>
            <p>Random forest: <span style='font-style: italic; color: #2980b9;'>{RANDOM_FOREST}</span></p>
            <p>Gradient boosted trees: <span style='font-style: italic; color: #2980b9;'>{GRADIENT_BOOSTING}</span></p>
        </div>
    </section>
</div>
"""

display(HTML(html_content))

In [ ]:
PREPROCESSING_ERROR = False

In [ ]:
try:
    # Load dataset
    df = pd.read_json(DATASET)
    df.index.name = "index"
    classes = df[CLASS_ATTRIBUTE_EVENT].value_counts()
    samples_per_class = min(classes.min(), 5)
    sample = df.groupby(CLASS_ATTRIBUTE_EVENT).apply(lambda x: x.sample(samples_per_class)).reset_index(drop=True)
    selected_indices = sample.index
    display(md(f"## Dataset sample"))
    show(sample.reset_index(drop=True), show_index=False)

    # Preprocessing
    df.columns = [str(col) for col in df.columns]
    original_features = df.columns.drop([CLASS_ATTRIBUTE_EVENT, CLASS_ATTRIBUTE_TIME])
    # cast survival status to boolean
    df[CLASS_ATTRIBUTE_EVENT] = df[CLASS_ATTRIBUTE_EVENT].astype(int).astype(bool)
    display(md(f"## Preprocessed dataset"))
    
    if UNIMPORTANT_ATTRIBUTE_ELIMINATION:
        ISMT_THRESHOLD = {'I': I, 'S': T, 'M': M, 'T': T}
        df,  low_quality_details= remove_low_quality_attributes(df, [CLASS_ATTRIBUTE_EVENT, CLASS_ATTRIBUTE_TIME], ISMT_THRESHOLD)
        display(md("#### Removed Attributes"))
        reasons_combined = {**low_quality_details}
        for reason, details in reasons_combined.items():
            if details is not None and len(details) > 0:
                if isinstance(details, pd.Index):
                    details = details.tolist()  
                details_str = ', '.join(details)
            else:
                details_str = 'None'
            display(md(f"**{reason}:** {details_str}"))
    
    if MISSING_VALUES_IMPUTATION:
        df = impute_missing_values(df, [CLASS_ATTRIBUTE_EVENT, CLASS_ATTRIBUTE_TIME], STRATEGY_NUMERICAL, STRATEGY_NOMINAL)
    else:
        df = df.dropna()
    
    display(md("#### Dataset after preprocessing"))
    preprocessed_sample = df.loc[selected_indices].reset_index(drop=True)
    show(preprocessed_sample, show_index=False)
    display(md("<br>"))
    display(md("##### Note: uncensored cases at the highest survival time will be excluded from the analysis"))
    models = []
    
    # X and y SPLIT
    _, X, _, y = preprocess_survival_data(df, CLASS_ATTRIBUTE_EVENT, CLASS_ATTRIBUTE_TIME)
    # TT / CV SPLIT
    if ANALYSIS_MODE == PredictionSettings.AnalysisMode.TRAIN_TEST:
        split = SurvivalStratifiedShuffleSplit(stratify_by=CLASS_ATTRIBUTE_EVENT, time_attribute=CLASS_ATTRIBUTE_TIME, n_splits=1, test_size=TEST_SIZE, random_state=42)
    else:
        split = SurvivalStratifiedKFold(stratify_by=CLASS_ATTRIBUTE_EVENT, time_attribute=CLASS_ATTRIBUTE_TIME, n_splits=5, shuffle=True, random_state=42)
    
    common_params = {
        "scoring": SCORING,
        "grid_search": GRID_SEARCH,
        "X": X,
        "y": y,
        "split": split,
        "mode": ANALYSIS_MODE,
        "original_features": original_features,
        "class_attribute_time": CLASS_ATTRIBUTE_TIME,
        "class_attribute_event": CLASS_ATTRIBUTE_EVENT,
    }
except Exception as e:
    logger.exception(e)
    display(md("An error occurred during preprocessing. No models will be trained."))
    PREPROCESSING_ERROR = True

In [ ]:
def show_coefs(regression_model):
    columns = ["Coefficients"]
    coefs = pd.DataFrame(regression_model.coef_.T, index=X.columns, columns=columns)
    display(md(f"#### Coefficients:"))
    show(coefs)


if COX_PROPORTIONAL_HAZARD and not PREPROCESSING_ERROR:
    from sksurv.linear_model import CoxPHSurvivalAnalysis
    model_class = CoxPHSurvivalAnalysis
    model_name = "Cox Proportional Hazards"
    fixed_grid = {
        "alpha": 0.5,
    }
    param_grid = {
        "tol": [1e-9, 1e-6, 1e-3],
        "ties": ["efron", "breslow"],
    }
    
    manager = SurvivalManager(
        model_class=model_class,
        model_name=model_name,
        fixed_grid=fixed_grid,
        param_grid=param_grid,
        **common_params,
        additional_method=show_coefs,
    )
    manager.run()
    models.append(manager)

In [ ]:
if DECISION_TREE and not PREPROCESSING_ERROR:
    from sksurv.tree import SurvivalTree
    model_class = SurvivalTree
    model_name = "Survival Decision Tree"
    fixed_grid = {}
    param_grid = {
        "splitter": ["best", "random"],
        "max_depth": [None, 3, 5, 10],
        "min_samples_split": [0.01, 0.02, 0.05],
    }
    
    manager = SurvivalManager(
        model_class=model_class,
        model_name=model_name,
        fixed_grid=fixed_grid,
        param_grid=param_grid,
        **common_params,
    )
    manager.run()
    models.append(manager)

In [ ]:
if RANDOM_FOREST and not PREPROCESSING_ERROR:
    from sksurv.ensemble import RandomSurvivalForest
    model_class = RandomSurvivalForest
    model_name = "Random Survival Forest"
    fixed_grid = {}
    param_grid = {
        "n_estimators": [100, 1000],
        "max_depth": [None, 3, 5, 10],
        "min_samples_split": [0.01, 0.02, 0.05],
    }
    
    manager = SurvivalManager(
        model_class=model_class,
        model_name=model_name,
        fixed_grid=fixed_grid,
        param_grid=param_grid,
        **common_params,
    )
    manager.run()
    models.append(manager)

In [ ]:
if SVM and not PREPROCESSING_ERROR:
    from sksurv.svm import FastSurvivalSVM
    model_class = FastSurvivalSVM
    model_name = "Linear Survival SVM"
    fixed_grid = {
        "max_iter": 100,
    }
    param_grid = {
        "alpha": [0.1, 0.5, 1.0],
        "rank_ratio": [0.01, 0.1, 0.5],
    }
    
    manager = SurvivalManager(
        model_class=model_class,
        model_name=model_name,
        fixed_grid=fixed_grid,
        param_grid=param_grid,
        **common_params,
        ibs=False,
    )
    manager.run()
    models.append(manager)

In [ ]:
if SVM and not PREPROCESSING_ERROR:
    from sksurv.svm import FastKernelSurvivalSVM
    model_class = FastKernelSurvivalSVM
    model_name = " Fast Kernel Survival SVM"
    fixed_grid = {
        "max_iter": 100,
    }
    param_grid = {
        "alpha": [0.1, 0.5, 1.0],
        "rank_ratio": [0.01, 0.1, 0.5],
        "kernel": ["rbf", ],
        "gamma": [0.01, 0.1],
    }
    
    manager = SurvivalManager(
        model_class=model_class,
        model_name=model_name,
        fixed_grid=fixed_grid,
        param_grid=param_grid,
        **common_params,
        ibs=False,
    )
    manager.run()
    models.append(manager)

In [ ]:
if GRADIENT_BOOSTING and not PREPROCESSING_ERROR:
    from sksurv.ensemble import GradientBoostingSurvivalAnalysis
    model_class = GradientBoostingSurvivalAnalysis
    model_name = "Survival Gradient Boosting"
    fixed_grid = {}
    param_grid = {
        "loss": ["coxph"],
        "n_estimators": [100, 200, 500],
        "learning_rate": [0.3, 0.4],
    }
    
    manager = SurvivalManager(
        model_class=model_class,
        model_name=model_name,
        fixed_grid=fixed_grid,
        param_grid=param_grid,
        **common_params,
    )
    manager.run()
    models.append(manager)

In [ ]:
if not PREPROCESSING_ERROR and len(models):
    summary = SurvivalSummarizer(models)
    summary.compare()